In [ ]:
import numpy as np 
import pandas as pd 
import os 
import sys 
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt 
# %matplotlib notebook
sys.path.insert(1,'../../../../arimaclust/')
from arimaclust.models.centroid import ConsistentARMAClustering
from arimaclust.models.neighbours import NearestNeighbourModel
from arimaclust.autoarima.estimation import estimate_processes
from arimaclust.metrics.matrix import dist_matrix_parallel
from arimaclust.tracking.mapping import mapp_hungarian

In [ ]:
df_usd = pd.read_csv('data/final_usd_2010_2020.csv',index_col='Unnamed: 0')

## Drop some symbols 

In [ ]:
# drop symbols with unknown behaviour (picewise constants)
to_drop = ['ZMK','BYR','LVL','TMT','CLF','ZWL','JEP','CUP','ERN','CUC','USD']
df_usd.drop(columns=to_drop,inplace=True)

# symbols with too much nans 
to_much_nans = ['BYN','IMP','GGP','ZMW','KYD']
df_usd.drop(columns=to_much_nans,inplace=True)

# metals 
metals = ['XAU','XAG']
df_usd.drop(columns=metals,inplace=True)

# coins 
coins = ['BTC']
df_usd.drop(columns=coins,inplace=True)


## We have nans in some series 

In [ ]:
nans = df_usd.isna().sum().sort_values(ascending=False)
nan_symb = nans.index[nans!=0]

In [ ]:
for ns in nan_symb:
    df_usd[ns].plot()
    
    plt.title(f'{ns}:{nans[ns]}')
    plt.show()

In [ ]:
df_usd[df_usd['IDR'].isna()]

## Fill with interpolation

In [ ]:

for ns in nan_symb:
    
    l,u = df_usd[df_usd[ns].isna()].index.min(),df_usd[df_usd[ns].isna()].index.max()
    inter = df_usd[ns].interpolate()  
    df_usd[l:u][ns].plot(style='r-',label='True:-')
    inter[l:u].plot(style='g--',label='Interpolated:*')
    plt.title(f'{ns}:{nans[ns]}')
    plt.legend()
    plt.show()

In [ ]:
df_usd = df_usd.interpolate()

## Some utils

In [ ]:
from pmdarima.preprocessing import BoxCoxEndogTransformer


def prepare_data(df,bc=True,ws=None):
    X = []
    for col in df.columns:
        if ws:
            x = df[col].rolling(ws).mean().dropna().values
        else:
            x = df[col].dropna().values
        if bc:
            print(col)
            x, _ = BoxCoxEndogTransformer(lmbda2=1e-6).fit_transform(x)

        X.append(x)
    return X

def plot_mds(dist_matrix,labels,names):
    from sklearn import manifold 

    mds = manifold.MDS(
        n_components=2,
        max_iter=3000,
        eps=1e-9,
        random_state=42,
        dissimilarity="precomputed",
        n_jobs=-1,
        normalized_stress="auto",
    )
    pos = mds.fit(dist_matrix).embedding_


    nmds = manifold.MDS(
        n_components=2,
        metric=False,
        max_iter=3000,
        eps=1e-12,
        dissimilarity="precomputed",
        random_state=42,
        n_jobs=-1,
        n_init=1,
        normalized_stress="auto",
    )
    pos = nmds.fit_transform(dist_matrix, init=pos)
    fig, ax = plt.subplots(figsize=(12,8))
    ax.scatter(pos[:,0],pos[:,1],c=labels)
    for i in range(len(names)):
        ax.annotate(names[i], (pos[i][0], pos[i][1]),fontsize=8)
        
def plot_clusters(df,labels,k):
    for i in range(k):
        df[df.columns[labels==i]].plot()
        plt.legend(bbox_to_anchor=(0, 1, 1, 0), ncol=2)
        plt.show()

## Training

In [ ]:
start,end = '2010-01-01','2020-01-01'
current_start,current_end = '2010-01-01','2011-01-01'

process_args = {'max_p':3,
               'max_q':3,
               'trace':True,
               'd':0
              }

metric_args = {'tau':30}
k = 10

df_usd_part1 = df_usd[current_start:current_end]
# df_usd_part1.drop(columns=[col for col in df_usd_part1.columns if df_usd_part1[col].nunique() == 1],inplace=True)

df_usd_part1 = np.log1p(df_usd_part1.pct_change())

X1 = prepare_data(df=df_usd_part1,bc=False,ws=2)
model_usd1 = ConsistentARMAClustering(k=k, metric = 'piccolo', metric_args=metric_args,process_args=process_args)
model_usd1.fit(X1)


In [ ]:
plot_mds(dist_matrix=model_usd1.dist_matrix,labels=model_usd1.labels_,names=df_usd_part1.columns)

In [ ]:
# df_usd_part_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_usd_part1),columns=df_usd_part1.columns)
plot_clusters(df_usd_part1,model_usd1.labels_,k=k)

In [ ]:
start,end = '2010-01-01','2020-01-01'
current_start,current_end = '2011-01-01','2012-01-01'

process_args = {'max_p':3,
               'max_q':3,
               'trace':True,
               'd':0
              }

metric_args = {'tau':30}
k = 10

df_usd_part_2 = df_usd[current_start:current_end]
# df_usd_part_2.drop(columns=[col for col in df_usd_part_2.columns if df_usd_part_2[col].nunique() == 1],inplace=True)
df_usd_part_2 = np.log1p(df_usd_part_2.pct_change())

X2 = prepare_data(df=df_usd_part_2, bc=False,ws=2)
model_usd2 = ConsistentARMAClustering(k=k, metric = 'piccolo', metric_args=metric_args,process_args=process_args)
model_usd2.fit(X2)


In [ ]:
plot_mds(model_usd2.dist_matrix,labels=model_usd2.labels_,names=df_usd_part_2.columns)

In [ ]:
# df_usd_part_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_usd_part_2),columns=df_usd_part_2.columns)
plot_clusters(df_usd_part_2,model_usd2.labels_,k=k)

In [ ]:
symb = 'CNY'

l1 = df_usd_part1.columns[model_usd1.labels_ == model_usd1.labels_[list(df_usd_part1.columns).index(symb)]]
l2 = df_usd_part_2.columns[model_usd2.labels_ == model_usd2.labels_[list(df_usd_part_2.columns).index(symb)]]

len(set(l1).intersection(set(l2)))/max(len(l1),len(l2))

In [ ]:
r,c = mapp_hungarian(model_usd1.labels_,model_usd2.labels_)

for i,j in zip(r,c):
    print(i,j)
    print(df_usd_part1.columns[model_usd1.labels_ == j],df_usd_part_2.columns[model_usd2.labels_ == i], '\n')

In [ ]:
from datetime import datetime
start,end = datetime(year=2010, day=1, month= 1),datetime(year=2020, day=1, month= 1)
current_start = start 
n_years = end.year - start.year
labels = []
dm = []

for i in range(n_years):
    current_end = datetime(year=current_start.year+1, day=1, month= 1)

    process_args = {'max_p':3,
               'max_q':3,
               'trace':True,
               'd':0
              }

    metric_args = {'tau':30}
    k = 10
    ws = 2

    df_part = df_usd[str(current_start):str(current_end)]
    # df_usd_part_2.drop(columns=[col for col in df_usd_part_2.columns if df_usd_part_2[col].nunique() == 1],inplace=True)
    df_part = np.log1p(df_part.pct_change())

    X = prepare_data(df=df_part, bc=False,ws=ws)
    model_part = ConsistentARMAClustering(k=k, metric = 'piccolo',
                                          metric_args=metric_args,process_args=process_args)
    model_part.fit(X)
    labels.append(model_part.labels_)
    dm.append(model_part.dist_matrix)
    
    current_start = current_end

In [ ]:
r,c = mapp_hungarian(labels[0],labels[1])

for i,j in zip(r,c):
    print(i,j)
    print(df_part.columns[labels[0] == i],df_part.columns[labels[1] == j], '\n')

In [ ]:
symb = 'AMD'
start,end = datetime(year=2010, day=1, month= 1),datetime(year=2020, day=1, month= 1)
current_start = start 
n_years = end.year - start.year

for i in range(n_years):
    current_end = datetime(year=current_start.year+1, day=1, month= 1)
    print(current_start,current_end)
    l1 = df_part.columns[labels[i] == labels[i][list(df_part.columns).index(symb)]]
    l2 = df_part.columns[labels[i+1] == labels[i+1][list(df_part.columns).index(symb)]]
    print(len(set(l1).intersection(set(l2)))/max(len(l1),len(l2)))
    current_start = current_end
    